# 1. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, regexp_replace

# 2. Read Bronze Table 

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
df.display()

# 3. Silver Layer Transformations

## 3.1. Trim Whitespace from All String Columns in Dataframe

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


## 3.2. Data Normalizastion

In [0]:
df = (
    df.withColumn(
        'cst_marital_status', 
        F.when(col('cst_marital_status') == 'S', 'Single')
        .when(col('cst_marital_status') == 'M', 'Married')

    ).withColumn(
        'cst_gndr', 
        F.when(col('cst_gndr') == 'M', 'Male')
        .when(col('cst_gndr') == 'F', 'Female')
        .otherwise('n/a')
    )
)   

In [0]:
df.display(10)

## 3.3. Filter DataFrame to Exclude Null Customer IDs

In [0]:
df = df.filter(col("cst_id").isNotNull())

## 3.4. Rename DataFrame Columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
    

In [0]:
# Sanity checks of dataframe
df.limit(10).display()

# 4. Writing Table To Silver Layer
 

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

In [0]:
%sql
-- Sanity Checks of Silver Table
SELECT * FROM silver.crm_customers LIMIT 10;